In [ ]:
%%writefile ambiente.h
// Contiene: i parametri, gli ostacoli (scatole, sfere, fori), la maschera della griglia,
// le sorgenti della passata 1 (bordo degli ostacoli)
#pragma once           // il file viene incluso una sola volta
#include <vector>
#include <cmath>
#include <algorithm>

// Forme con cui si descrive l'ambiente (coordinate nel cubo [-1, 1]^3)
struct Punto   { double x, y, z; };
struct Scatola { double x0, x1, y0, y1, z0, z1; };       // parallelepipedo: x0 <= x <= x1, y0 <= y <= y1, z0 <= z <= z1
struct Sfera   { double cx, cy, cz, r; };                 // centro (cx, cy, cz) e raggio r


// AMBIENTE: per cambiarlo si modifica solo questa parte
// =====================================================================
// Stanza = cubo [-1, 1]^3 (20 m di lato), z verticale (z = -1 pavimento). Le 6 pareti sono ostacoli (PARETI).
const double MARGINE = 0.06;                             // distanza minima dagli ostacoli (raggio del drone + sicurezza)
const double D_SICUREZZA = 0.3;                          // distanza dagli ostacoli oltre la quale si va a velocita' piena
const double V_MIN = 0.1;                                // velocita' minima, a ridosso degli ostacoli (> 0)

const Punto A = { -0.8, -0.1, -0.4 };                    // partenza del drone: meta' sinistra della stanza, in basso
const Punto B = {  0.75, 0.4, -0.1 };                    // arrivo (sorgente del FIM): meta' destra, piu' vicino all'apertura destra

// Oggetto mobile: un muro a x = 0 con due aperture (sinistra y < 0, destra y > 0). Un pannello chiude una sola apertura
// alla volta e si sposta sull'altra a intervalli regolari (il periodo si sceglie in Python): il drone deve sempre
// passare dall'apertura libera. Le pareti della stanza sono ostacoli anch'esse.
const double SP = 0.04;                                  // spessore delle pareti della stanza
const std::vector<Scatola> PARETI = {                    // le 6 pareti della stanza (pavimento e soffitto compresi)
    { -1.0, -1.0 + SP, -1.0, 1.0, -1.0, 1.0 }, { 1.0 - SP, 1.0, -1.0, 1.0, -1.0, 1.0 },
    { -1.0, 1.0, -1.0, -1.0 + SP, -1.0, 1.0 }, { -1.0, 1.0, 1.0 - SP, 1.0, -1.0, 1.0 },
    { -1.0, 1.0, -1.0, 1.0, -1.0, -1.0 + SP }, { -1.0, 1.0, -1.0, 1.0, 1.0 - SP, 1.0 },
};
const Scatola MURO = { -0.05, 0.05, -1.00, 1.00, -1.00, 1.00 };            // muro centrale (x = 0), spesso 1 m
const Scatola APERTURA_S = { -0.05, 0.05, -0.70, -0.30, -0.80, 0.40 };     // apertura sinistra (y < 0): 4 m x 12 m
const Scatola APERTURA_D = { -0.05, 0.05,  0.30,  0.70, -0.80, 0.40 };     // apertura destra (y > 0): 4 m x 12 m
const std::vector<Sfera> SFERE = {};                     // {cx, cy, cz, r}: nessuna sfera, per ora

// Le due mappe dell'oggetto mobile. imposta_mappa(m) sceglie la mappa attuale:
// 0 = pannello sull'apertura sinistra (aperta la destra), 1 = pannello sull'apertura destra (aperta la sinistra)
inline std::vector<Scatola> SCATOLE;
inline std::vector<Scatola> FORI;                        // zone in cui l'ostacolo viene tolto (l'apertura libera)
inline void imposta_mappa(int m)
{
    SCATOLE = PARETI;
    SCATOLE.push_back(MURO);
    FORI = { m == 0 ? APERTURA_D : APERTURA_S };         // il pannello chiude l'altra apertura
}
inline const bool MAPPA_INIZIALE = (imposta_mappa(0), true);   // all'avvio: mappa 0
// =====================================================================

// Vero se il punto (x, y, z) e' dentro la scatola s (bordi compresi)
inline bool dentro(const Scatola& s, double x, double y, double z)
{
    return s.x0 <= x && x <= s.x1 && s.y0 <= y && y <= s.y1 && s.z0 <= z && z <= s.z1;
}

// Vero se il punto (x, y, z) e' dentro la sfera s (distanza dal centro <= raggio)
inline bool dentro(const Sfera& s, double x, double y, double z)
{
    return (x - s.cx) * (x - s.cx) + (y - s.cy) * (y - s.cy) + (z - s.cz) * (z - s.cz) <= s.r * s.r;
}

// Vero se il punto (x, y, z) e' dentro un ostacolo (i FORI tolgono l'ostacolo)
inline bool ostacolo(double x, double y, double z)
{
    for (const auto& f : FORI)    if (dentro(f, x, y, z)) return false;
    for (const auto& s : SCATOLE) if (dentro(s, x, y, z)) return true;
    for (const auto& s : SFERE)   if (dentro(s, x, y, z)) return true;
    return false;
}

// Maschera della griglia: libero = 1, ostacolo = 0 (resta INF)
inline std::vector<unsigned char> costruisci_maschera(int N)
{
    const double h = 2.0 / (N - 1);
    const int n = N * N * N;
    std::vector<unsigned char> mask(n);
    for (int q = 0; q < n; ++q) {                     // un nodo alla volta
    const int j = q % N, i = (q / N) % N, k = q / (N * N);
    mask[q] = ostacolo(-1.0 + j * h, -1.0 + i * h, -1.0 + k * h) ? 0 : 1;
    }
    // Margine: diventa ostacolo ogni nodo entro MARGINE da un ostacolo
    const int r = (int)std::ceil(MARGINE / h);               // raggio sfera intorno all'ostacolo
    std::vector<unsigned char> margin_mask = mask;
    for (int q = 0; q < n; ++q) {
        if (mask[q]) continue;                                   
    const int k = q / (N * N), i = (q / N) % N, j = q % N;
    const bool bordo = (j > 0 && mask[q - 1]) || (j < N - 1 && mask[q + 1]) ||          // che toccano un nodo libero
                       (i > 0 && mask[q - N]) || (i < N - 1 && mask[q + N]) ||
                       (k > 0 && mask[q - N * N]) || (k < N - 1 && mask[q + N * N]);
    if (!bordo) continue;
    for (int dk = -r; dk <= r; ++dk)                          // tutti i nodi entro la sferadi raggio MARGINE
        for (int di = -r; di <= r; ++di)
            for (int dj = -r; dj <= r; ++dj) {
                const int kk = k + dk, ii = i + di, jj = j + dj;
                if (kk < 0 || kk >= N || ii < 0 || ii >= N || jj < 0 || jj >= N) continue;
                if ((dk * dk + di * di + dj * dj) * h * h <= MARGINE * MARGINE)
                    margin_mask[(kk * N + ii) * N + jj] = 0;
            }
}
return margin_mask;
}
// Sorgenti della passata 1: i nodi liberi che toccano un ostacolo
inline std::vector<int> nodi_di_bordo(const std::vector<unsigned char>& mask, int N)
{
    std::vector<int> bordo;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
                const int q = (k * N + i) * N + j;
                if (!mask[q]) continue;                                  // solo nodi liberi...
                if ((j > 0 && !mask[q - 1]) || (j < N - 1 && !mask[q + 1]) ||
                    (i > 0 && !mask[q - N]) || (i < N - 1 && !mask[q + N]) ||
                    (k > 0 && !mask[q - N * N]) || (k < N - 1 && !mask[q + N * N]))
                    bordo.push_back(q);                                  // ...che hanno un vicino ostacolo
            }
    return bordo;
}

// FM2: dalla distanza d dagli ostacoli alla lentezza F = 1/v della passata 2
// v = V_MIN + (1 - V_MIN) * min(d / D_SICUREZZA, 1): F = 10 sul bordo, 1 lontano
inline std::vector<float> lentezza_fm2(const std::vector<float>& d)
{
    std::vector<float> F(d.size());
    for (size_t q = 0; q < d.size(); ++q) {
        const double r = std::min((double)d[q] / D_SICUREZZA, 1.0);       // distanza normalizzata: 0 sul bordo, 1 oltre D_SICUREZZA (o se d = INF)
        F[q] = (float)(1.0 / (V_MIN + (1.0 - V_MIN) * r));
    }
    return F;
}
// Controllo iniziale: A e B non devono stare dentro un ostacolo
inline bool ambiente_valido()
{
    return !ostacolo(A.x, A.y, A.z) && !ostacolo(B.x, B.y, B.z);
}

// Dal punto (x, y, z) all'indice del nodo piu' vicino (serve per A e B)
int nodo(int N, double x, double y, double z){
    const double h = 2.0 / (N - 1);
    const int j = (int)lround((x + 1.0) / h), i = (int)lround((y + 1.0) / h), k = (int)lround((z + 1.0) / h);
    return (k * N + i) * N + j;
}

In [ ]:
%%writefile fmm3d.h
#pragma once           // il file viene incluso una sola volta
#include <cstdio>    
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include <chrono>      // misura dei tempi
#include <fstream>     // scrittura su file
const float INF = 1e30f;                 // "infinito": nodo non ancora raggiunto dall'onda
typedef std::pair<float, int> Entry;     // coppia (u, indice del nodo) per la coda di priorita' del FMM

// Risolutore locale 3D (Godunov, Alg. 2 esteso al 3D): dai minimi dei vicini a, b, c al nuovo u
// Prova con 1, 2, poi 3 vicini; usato sia dal FMM (CPU) sia dal Block FIM (GPU)
__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;                              // tempo per attraversare una cella
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

// Nuovo u del nodo (k, i, j) per il FMM: minimo dei vicini per asse + risolutore locale
float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int k, int i, int j, int N, float h)
{
    int idx = (k * N + i) * N + j;  // indice globale del nodo (k, i, j)
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[idx - 1] : INF;
    float right = (j < N - 1) ? u[idx + 1] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[idx - N] : INF;
    float up    = (i < N - 1) ? u[idx + N] : INF;
    // c: min dei vicini lungo z (stessa colonna e riga, piano k-1 e k+1)
    float back  = (k > 0)     ? u[(idx - N * N)] : INF;
    float front = (k < N - 1) ? u[(idx + N *N)] : INF;

    float a = fminf(left, right);
    float b = fminf(down, up);
    float c = fminf(back, front);

    float q = solve_quadratic3(a, b, c, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ijk, u)
}
// Fast Marching Method (Alg. 3) su CPU: il riferimento per il Block FIM
// Fissa i nodi in ordine di tempo crescente con una coda di priorita'
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, const std::vector<int>& srcs, const std::vector<unsigned char>& mask)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N * N, LONTANO);    // array di char NxNxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    for (int s : srcs){heap.push(Entry(u[s], s)); state[s] = ATTIVO;}

    const int dk[6] = {  0, 0,  0, 0, -1, 1 };       // spostamenti verso i 6 vicini:
    const int di[6] = {  0, 0, -1, 1,  0, 0 };       // sinistra, destra, sotto, sopra,
    const int dj[6] = { -1, 1,  0, 0,  0, 0 };       // davanti, dietro

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int k = x / (N*N), i = (x / N) % N, j = x % N;                    // strato, riga, colonna del nodo Trial
        for (int d = 0; d < 6; ++d) {                // foreach x_nb adiacente a Trial
            int ni = i + di[d], nj = j + dj[d], nk = k + dk[d];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N || nk < 0 || nk >= N) continue;   // fuori dal dominio
            int nb = (nk * N + ni) * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO || !mask[nb]) continue;         // if x_nb non appartiene a Set

            float q = solve_node(u, F, nk, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}

In [ ]:
%%writefile block_fim3d.cuh
#pragma once           // il file viene incluso una sola volta
// Block FIM 3D (Alg. 7) sulla GPU: blocchi 8^3 in shared memory, uscita anticipata, maschera degli ostacoli
#include <cstdio>
#include <cstdlib>
#include <vector>
#include <algorithm>       // std::min
#include <cuda_runtime.h>
#include "fmm3d.h"         // INF e solve_quadratic3 (lo stesso risolutore locale del FMM)

// Controllo degli errori CUDA: se una chiamata fallisce, stampa dove e perche' ed esce
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

#define BLOCK 8  // blocco 8 x 8 x 8 = 512 nodi, un thread per nodo
// Risolutore locale in shared memory: i 6 vicini del nodo sono nella copia s (blocco + halo)
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}

// Step 1: niter iterazioni; Step 2: 1 iterazione di prova.
__device__ void process_block(int block, int niter, float* u, const float* F,
                             int N, int nbx, float h, const unsigned char* mask, float eps, unsigned char* Cb_out)
{
    // Allocazione statica di shared memory
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];              
    __shared__ bool s_Cn[BLOCK * BLOCK * BLOCK];                  // i flag Cn dei 512 nodi
    // posizione del thread nel suo blocco CUDA: ognuno da 0 a 7
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    // bj: asse x , bi: asse y e bk: asse z; identifica blocco nella griglia
    int bj = block % nbx, bi = (block / nbx) % nbx, bk = block / (nbx * nbx); 
    // numero del thread dentro il blocco; 
    int tid = (tz*BLOCK + ty) * BLOCK + tx; // 0, ..., 511
    // posizione globale rispetto x,y e z: lungo tutta la griglia
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;
    int idx = (k * N + i) * N + j;  // 0, ..., N^3-1
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || !mask[idx];             // nodo da non aggiornare: fuori dalla griglia o maschera
    float f = inside ? F[idx] : 1.0f;
    // Global -> Shared
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        // posizione nella shared memory
        int sz = m / ((BLOCK + 2) * (BLOCK + 2)), sy = (m / (BLOCK + 2)) % (BLOCK + 2), sx = m % (BLOCK + 2);
        // coordinate globali della cella (sz, sy, sx): -1 perché c'è l'halo                          
        int gi = bi * BLOCK + sy - 1, gj = bj * BLOCK + sx - 1, gk = bk * BLOCK + sz - 1;
        
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();
    bool conv = true;
    for (int it = 0; it < niter; ++it){
        // p: valore del nodo all'inizio di questa iterazione
        float p = s_u[tz+1][ty+1][tx+1];
        // q: nuovo valore calcolato in questa iterazione
        float q;
        if (fixed) {q = p;}
        else{
        // tx, ty, tz: 0,...,7; +1: nella shared memory c'è l'halo
        q = solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);
        }    
        __syncthreads();
        s_u[tz+1][ty + 1][tx + 1] = q;
        conv = fabsf(p - q) < eps;
        if (!__syncthreads_or(!conv)) break;   // tutti hanno scritto; se nessun nodo e' cambiato, esci
    }
    // Shared -> Global: ogni thread riscrive il proprio nodo
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];
    // Cn del nodo: vero se non è cambiato nell'ultima iterazione
    s_Cn[tid] = conv;
    __syncthreads();
    for (int reduct = BLOCK * BLOCK * BLOCK / 2; reduct > 0; reduct /= 2) {     // 256, ..., 4, 2, 1
        if (tid < reduct) s_Cn[tid] = s_Cn[tid] && s_Cn[tid + reduct];
        __syncthreads();
    }
    if (tid == 0) Cb_out[block] = s_Cn[0];
}
// Step1: ogni blocco della lista A fa niter iterazioni
__global__ void update_active_blocks(const int* active_list, float* u, const float* F,
                                    int N, int nbx, float h, const unsigned char* mask, int niter,
                                    float eps, unsigned char* Cb_step1)
{
    // blockIdx è il numero del blocco CUDA nel lancio in corso
    process_block(active_list[blockIdx.x], niter, u, F, N, nbx, h, mask, eps, Cb_step1);
    // (es. N = 33) giro: 1 -> blocchi CUDA lanciati: 1 -> blockIdx.x: 0 -> block = active_list[blockIdx.x]: 62
    // (es. N = 33) giro: 3 -> blocchi CUDA lanciati: 6 -> blockIdx.x: 0, 1, ..., 5 -> block = active_list[blockIdx.x]: 61, 63, 57, 67, 37, 87
}

// costruisce la lista A dei blocchi attivi (Cb = 0)
__global__ void build_active_list(const unsigned char* Cb, unsigned char* in_list,
                                  int* active_list, int* n_active, int n_blocks) 
{
    // blockIdx: blocco nel lancio in corso; blockDim (monodimensionale CUDA): 256; threadIdx:0,...,255
    int b = blockIdx.x * blockDim.x + threadIdx.x;           // numero del thread(blocco) su tutto il lancio
    // N = 65 -> nbx = 9 -> n_blocks = 729 -> list_blocks = 3 -> thread lanciati = 768 -> thread in eccesso = 39 
    if (b >= n_blocks) return;                               // thread(blocco) in eccesso lanciati da CUDA                                   
    in_list[b] = !Cb[b];                                      // fotografia della lista A di questo giro
    if (!Cb[b]) active_list[atomicAdd(n_active, 1)] = b;               // se NON convergente aggiungi thread(blocco) alla lista A
}
// Step 2: i 6 vicini dei blocchi appena convergenti diventano candidati
__global__ void build_candidate_list(const int* active_list, int n_active, int nbx, const unsigned char* in_list,
                                     const unsigned char* Cb_step1, unsigned char* Cb,
                                     int* is_candidate, int* cand_list, int* cand_count)
{
    int pos = blockIdx.x * blockDim.x + threadIdx.x;
    // se è maggiore o uguale al numero di blocchi attivi fai return
    if (pos >= n_active) return;                       
    int b = active_list[pos];         // un blocco della lista A; N = 33 -> A ={61, 63, ...,87}; active_list[0] =61
    Cb[b] = Cb_step1[b];             // leggi il valore di Cb
    if (!Cb_step1[b]) return;        // proseguono solo i blocchi appena convergenti
    // bj: blocco lungo x; bi: blocco lungo y; bk: blocco lungo z
    int bj = b % nbx, bi = (b / nbx) % nbx, bk = b / (nbx * nbx);
    // sinistra, destra, sotto, sopra e davanti, dietro
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};
    // verifico se si trova nella griglia
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // scarta blocchi fuori dalla griglia
        // scarta blocchi attivi e non convergenti
        if (in_list[nb[d]] && !Cb_step1[nb[d]]) continue;          // d: indica la direzione; N = 33; nb[d] = [61, 63, 57, 67, 37, 87];      
        if (atomicExch(&is_candidate[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}
// Step 2: ogni candidato fa 1 iterazione; se cambia, diventa attivo
__global__ void update_candidate_blocks(const int* cand_list, const int* cand_count, int* is_candidate,
                                        float* u, const float* F, int N, int nbx, float h,
                                        const unsigned char* mask, float eps, unsigned char* Cb)
{
    // scarta blocchi CUDA in eccesso
    if (blockIdx.x >= *cand_count) return;
    int b = cand_list[blockIdx.x];
    // blocchi candidati: niter = 1 -> con la nuova informazione dal blocco convergente, questo vicino cambia?
    process_block(b, 1, u, F, N, nbx, h, mask, eps, Cb);
    if (threadIdx.x == 0 && threadIdx.y == 0 && threadIdx.z == 0) is_candidate[b] = 0;   // pronto per il giro successivo
}
// Ogni giro: lista A -> Step 1 -> Step 2; fine quando A e' vuota
int block_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
              const std::vector<int>& srcs, const std::vector<unsigned char>& mask, float eps, int niter, float* ms)
{
    // nbx: numero di blocchi per lato; n_blocks: blocchi totali
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx;
    // bytes: dimensione totale griglia 3D
    const size_t bytes = (size_t)N * N * N * sizeof(float);
    // Cb: si inizializza vettore di n_blocks valori a 1
    std::vector<unsigned char> Cb(n_blocks, 1);
    // strato, riga e colonna del nodo sorgente
    for (int s : srcs){
    const int sk = s / (N * N), si = (s / N) % N, sj = s % N;
    // Cb: vale 1 tranne nel blocco della sorgente (0) 
    Cb[((sk / BLOCK) * nbx + si / BLOCK) * nbx + sj / BLOCK] = 0;
    }

    // Memoria sulla GPU e copia dei dati iniziali (CPU -> GPU)
    float *d_u, *d_F; unsigned char *d_Cb, *d_Cb_step1, *d_in_list, *d_mask; int *d_active_list, *d_n_active;
    int *d_is_candidate, *d_cand_list, *d_cand_count;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_Cb_step1, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_in_list, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_active_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_n_active, sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_mask, N*N*N));
    // Memoria per i blocchi candidati
    CUDA_CHECK(cudaMalloc(&d_is_candidate, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));
    // si copiano d_u, d_F e d_Cb dalla CPU alla GPU
    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb.data(), n_blocks, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), N*N*N, cudaMemcpyHostToDevice));
    // si azzerano d_Cb_step1 e d_is_candidate
    CUDA_CHECK(cudaMemset(d_Cb_step1, 0, n_blocks));
    CUDA_CHECK(cudaMemset(d_is_candidate, 0, n_blocks * sizeof(int)));
    // list_blocks: blocchi CUDA da 256 threads per avere un thread per ogni blocco di nodi 
    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;
    // si misura l'ora prima del ciclo
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                     
    int iter = 0;
    while (true) {
        // azzera il numero di blocchi attivi                                           
        int n_active = 0;  // contatore su CPU
        CUDA_CHECK(cudaMemset(d_n_active, 0, sizeof(int)));    // contatore su GPU
        // lancia list_blocks CUDA da 256 threads ciascuno, sufficiente per avere un thread ogni blocco
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_in_list, d_active_list, d_n_active, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        // aggiorna n_active con d_n_active
        CUDA_CHECK(cudaMemcpy(&n_active, d_n_active, sizeof(int), cudaMemcpyDeviceToHost));
        if (n_active == 0) break;  // se n_active == 0, lista A vuota, termina ciclo
        ++iter;                 // conta i giri
        // Step 1: foreach b in A in parallelo; lancia n_active blocchi 8x8x8
        update_active_blocks<<<n_active, dim3(BLOCK, BLOCK, BLOCK)>>>(d_active_list, d_u, d_F, N, nbx, h, d_mask, niter, eps, d_Cb_step1);    // si lanciano n_active blocchi CUDA, ognuno con 512 thread
        CUDA_CHECK(cudaGetLastError());
        // azzeriamo il contatore dei candidati sulla GPU
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        // lancia n_active blocchi CUDA da 256 threads
        build_candidate_list<<<(n_active + threads-1) / threads, threads>>>(d_active_list, n_active, nbx, d_in_list, d_Cb_step1, d_Cb,
                                                                          d_is_candidate, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        // ogni blocco ha al più 6 vicini, quindi 6*n_active
        update_candidate_blocks<<<std::min(6 * n_active, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, d_is_candidate,
                                                                                    d_u, d_F, N, nbx, h, d_mask, eps, d_Cb);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    // Risultato sulla CPU (GPU -> CPU) e liberazione della memoria
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_Cb_step1)); CUDA_CHECK(cudaFree(d_mask));
    CUDA_CHECK(cudaFree(d_in_list)); CUDA_CHECK(cudaFree(d_active_list)); CUDA_CHECK(cudaFree(d_n_active));
    CUDA_CHECK(cudaFree(d_is_candidate)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;
}


In [ ]:
%%writefile main.cu
// FM2 (passata 1 + passata 2) con il FMM e con il Block FIM: tabella al variare di N e ripianificazione con un oggetto mobile
#include <cstdio>
#include <algorithm>       // std::min
#include <chrono>          // tempi sulla CPU
#include <fstream>         // salvataggio su file
#include <string>          // nomi dei file delle mappe
#include <cuda_runtime.h>  // prima di fmm3d.h: definisce __host__ __device__
#include "fmm3d.h"         // Fast Marching (CPU)
#include "ambiente.h"      // stanza, ostacoli, maschera, lentezza del FM2
#include "block_fim3d.cuh" // Block FIM (GPU)

// FM2 completo (passata 1 + passata 2) sulla mappa attuale, con il FMM (CPU) o con il Block FIM (GPU).
// Restituisce u (tempo di arrivo da B), la F del FM2 e il tempo di calcolo delle due passate [ms].
void fm2(bool con_block_fim, int Nk, const std::vector<unsigned char>& mask, float eps, int niter,
         std::vector<float>& u, std::vector<float>& Fk, double& t_ms)
{
    const float hk = 2.0f / (Nk - 1);
    const int n = Nk * Nk * Nk;
    // prima passata: numero di nodi liberi che toccano un ostacolo
    const std::vector<int> bordo = nodi_di_bordo(mask, Nk);
    // seconda passata: percorso minimo per B          
    const std::vector<int> srcs = { nodo(Nk, B.x, B.y, B.z) };          
    std::vector<float> d(n, INF);
    for (int s : bordo) d[s] = 0.0f;
    const std::vector<float> uno(n, 1.0f);
    u.assign(n, INF);
    for (int s : srcs) u[s] = 0.0f;
    if (con_block_fim) {                                     // GPU: tempo del solo ciclo dei giri (come nelle tabelle)
        float ms1 = 0.0f, ms2 = 0.0f;
        // calcolo la distanza dei nodi liberi dagli ostacoli
        block_fim(d, uno, Nk, hk, bordo, mask, eps, niter, &ms1);
        // definisco la lentezza
        Fk = lentezza_fm2(d);
        // tempo di arrivo in B
        block_fim(u, Fk, Nk, hk, srcs, mask, eps, niter, &ms2);
        // tempo impiegato dal FM2 con Block FIM
        t_ms = ms1 + ms2;
    } else {                                                 // CPU: le due chiamate al FMM
        auto t0 = std::chrono::high_resolution_clock::now();
        fmm(d, uno, Nk, hk, bordo, mask);
        Fk = lentezza_fm2(d);
        fmm(u, Fk, Nk, hk, srcs, mask);
        auto t1 = std::chrono::high_resolution_clock::now();
        t_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();
    }
}

// Scrive i file di una mappa per Python: u, maschera, F e ambiente (ostacoli e fori della mappa attuale)
void scrivi_mappa(int Nk, int m, const std::vector<float>& u, const std::vector<unsigned char>& mask, const std::vector<float>& Fk)
{
    const std::string t = std::to_string(Nk) + "_mappa" + std::to_string(m);
    std::ofstream out("fim_" + t + ".bin", std::ios::binary);
    out.write(reinterpret_cast<const char*>(u.data()), u.size() * sizeof(float));
    std::ofstream om("mask_" + t + ".bin", std::ios::binary);
    om.write(reinterpret_cast<const char*>(mask.data()), mask.size());
    std::ofstream of("F_" + t + ".bin", std::ios::binary);
    of.write(reinterpret_cast<const char*>(Fk.data()), Fk.size() * sizeof(float));
    std::ofstream amb("ambiente_mappa" + std::to_string(m) + ".txt");
    amb.precision(15);
    amb << "MARGINE " << MARGINE << "\n";
    amb << "A " << A.x << " " << A.y << " " << A.z << "\n";
    amb << "B " << B.x << " " << B.y << " " << B.z << "\n";
    for (const auto& s : SCATOLE) amb << "SCATOLA " << s.x0 << " " << s.x1 << " " << s.y0 << " " << s.y1 << " " << s.z0 << " " << s.z1 << "\n";
    for (const auto& s : SFERE)   amb << "SFERA " << s.cx << " " << s.cy << " " << s.cz << " " << s.r << "\n";
    for (const auto& f : FORI)    amb << "FORO " << f.x0 << " " << f.x1 << " " << f.y0 << " " << f.y1 << " " << f.z0 << " " << f.z1 << "\n";
}

int main()
{
    // Informazioni sulla GPU usata
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d | max thread per blocco: %d | shared memory per blocco: %zu KB\n",
           prop.name, prop.multiProcessorCount, prop.maxThreadsPerBlock, prop.sharedMemPerBlock / 1024);
    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza
    const int niter = 8;                                     // iterazioni locali massime (uscita anticipata): la migliore con gli ostacoli (prove 12 / 24)

    if (!ambiente_valido()) {                                // A o B dentro un ostacolo: il problema non ha senso
        printf("Errore: A o B sono dentro un ostacolo, correggi ambiente.h\n");
        return 1;
    }
    printf("    N   giri   max|FIM - FMM|   u(A) FIM   u(A) FMM   t FMM [ms]   t FIM [ms]   speedup\n");

    for (int Nk : { 129, 193, 257, 385 }) {
        const int ripetizioni = 3;                           // ogni tempo = il minimo di 3 misure...
        const int ripetizioni_fmm = (Nk >= 385) ? 1 : 3;     // ...tranne il FMM a N = 385: dura minuti e varia di pochi percento
        const float hk = 2.0f / (Nk - 1);                    // passo della griglia nel cubo[-1, 1]^3
        const int n = Nk * Nk * Nk;                          // numero di nodi
        const std::vector<unsigned char> mask = costruisci_maschera(Nk);    // 1 = libero, 0= ostacolo
        const std::vector<int> srcs = { nodo(Nk, B.x, B.y, B.z) };          // la sorgente e' l'arrivo B
        const int iA = nodo(Nk, A.x, A.y, A.z);              // nodo della partenza A

        // passata 1: distanza dagli ostacoli (F = 1), con FIM e FMM
        const std::vector<int> bordo = nodi_di_bordo(mask, Nk);
        std::vector<float> d0(n, INF);                       // condizione iniziale: 0 sulbordo, INF altrove
        for (int s : bordo) d0[s] = 0.0f;
        const std::vector<float> uno(n, 1.0f);               // F = 1: u = distanza
        std::vector<float> d_ost, d_fmm;
        double t1_fmm = 1e30, t1_fim = 1e30;
        for (int r = 0; r < ripetizioni; ++r) {              // 3 misure, si tiene la piu'veloce
            if (r < ripetizioni_fmm) {                       // il FMM: 1 sola misura a N = 385
                d_fmm = d0;
                auto t0 = std::chrono::high_resolution_clock::now();
                fmm(d_fmm, uno, Nk, hk, bordo, mask);            // CPU: passata 1 con il FMM
                auto t1 = std::chrono::high_resolution_clock::now();
                t1_fmm = std::min(t1_fmm, std::chrono::duration<double, std::milli>(t1 - t0).count());
            }

            d_ost = d0;
            float ms = 0.0f;
            block_fim(d_ost, uno, Nk, hk, bordo, mask, eps, niter, &ms);   // GPU: passata 1 con il FIM
            t1_fim = std::min(t1_fim, (double)ms);
        }
        float diff1 = 0.0f;                                  // la distanza del FIM devecoincidere con quella del FMM
        for (int q = 0; q < n; ++q) diff1 = fmaxf(diff1, fabsf(d_ost[q] - d_fmm[q]));

        // FM2: lentezza F dalla distanza dagli ostacoli
        const std::vector<float> Fk = lentezza_fm2(d_ost);
        // passata 2: tempo di arrivo da B con questa F, con FMM e FIM
        std::vector<float> u0(n, INF);                       // condizione iniziale: 0 in B, INF altrove
        for (int s : srcs) u0[s] = 0.0f;

        std::vector<float> uk_fmm, uk_fim;
        double t_fmm = 1e30, t_fim = 1e30;
        int it = 0;
        for (int r = 0; r < ripetizioni; ++r) {
            if (r < ripetizioni_fmm) {                       // il FMM: 1 sola misura a N = 385
                uk_fmm = u0;                                     // ogni misura riparte da capo
                auto t0 = std::chrono::high_resolution_clock::now();
                fmm(uk_fmm, Fk, Nk, hk, srcs, mask);             // CPU: passata 2 con il FMM (Alg. 3)
        auto t1 = std::chrono::high_resolution_clock::now();
                t_fmm = std::min(t_fmm, std::chrono::duration<double, std::milli>(t1 - t0).count());
            }

            uk_fim = u0;
            float ms = 0.0f;
            it = block_fim(uk_fim, Fk, Nk, hk, srcs, mask, eps, niter, &ms);   // GPU:passata 2 con il Block FIM (Alg. 7)
            t_fim = std::min(t_fim, (double)ms);
        }

        // Controlli: FIM = FMM, nessun NaN, ostacoli a INF
        float diff = diff1;
        int n_nan = 0, ost_non_inf = 0;
        for (int q = 0; q < n; ++q) {
            diff = fmaxf(diff, fabsf(uk_fim[q] - uk_fmm[q]));
            if (std::isnan(uk_fim[q])) ++n_nan;
            if (!mask[q] && uk_fim[q] < INF) ++ost_non_inf;
        }
        if (n_nan > 0 || ost_non_inf > 0)                    // si stampa solo se qualcosa non va
            printf("ATTENZIONE (N = %d): %d valori NaN, %d nodi ostacolo raggiuntidall'onda\n", Nk, n_nan, ost_non_inf);

        const double t_fmm_tot = t1_fmm + t_fmm, t_fim_tot = t1_fim + t_fim;   // FM2 completo
        if (uk_fim[iA] >= INF)                               // l'onda partita da B non e' mai arrivata ad A
            printf("%5d   %4d   %14.2e   A NON RAGGIUNGIBILE da B\n", Nk, it, diff);
        else
            printf("%5d   %4d   %14.2e   %8.4f   %8.4f   %10.1f   %10.2f   %7.1f\n",
                   Nk, it, diff, uk_fim[iA], uk_fmm[iA], t_fmm_tot, t_fim_tot, t_fmm_tot / t_fim_tot);

        if (Nk == 129) {                                      // per i grafici (Python): soluzione, maschera e ambiente
            std::ofstream out("fim_129_ostacoli.bin", std::ios::binary);
            out.write(reinterpret_cast<const char*>(uk_fim.data()), uk_fim.size() * sizeof(float));
            std::ofstream om("mask_129.bin", std::ios::binary);
            om.write(reinterpret_cast<const char*>(mask.data()), mask.size());
            std::ofstream amb("ambiente.txt");               // l'ambiente di ambiente.h, una riga per elemento
            amb.precision(15);                               // 15 cifre: i numeri arrivano a Python esattamente come in ambiente.h
            amb << "MARGINE " << MARGINE << "\n";
            amb << "A " << A.x << " " << A.y << " " << A.z << "\n";
            amb << "B " << B.x << " " << B.y << " " << B.z << "\n";
            for (const auto& s : SCATOLE) amb << "SCATOLA " << s.x0 << " " << s.x1 << " " << s.y0 << " " << s.y1 << " " << s.z0 << " " << s.z1 << "\n";
            for (const auto& s : SFERE)   amb << "SFERA " << s.cx << " " << s.cy << " " << s.cz << " " << s.r << "\n";
            for (const auto& f : FORI)    amb << "FORO " << f.x0 << " " << f.x1 << " " << f.y0 << " " << f.y1 << " " << f.z0 << " " << f.z1 << "\n";
            std::ofstream of("F_129.bin", std::ios::binary);  // la lentezza F usata nellapassata 2 (serve a Python)
            of.write(reinterpret_cast<const char*>(Fk.data()), Fk.size() * sizeof(float));
        }
    }


    // RIPIANIFICAZIONE (N = 385) con l'oggetto mobile: il pannello passa da un'apertura all'altra durante il volo
    // (0 = pannello a sinistra, 1 = pannello a destra). Per ogni mappa: FM2 completo con FMM e Block FIM, tempi di ricalcolo
    // e file per l'animazione in Python
    {
        const int Nr = 385;                                  // griglia della ripianificazione (h = 0.0052, cioe' 5.2 cm in una stanza di 20 m)
        const int ripetizioni = 3, ripetizioni_fmm = 1;      // 3 misure per il Block FIM, 1 per il FMM (a N = 385 dura quasi un minuto)
        std::ofstream tr("tempi_ripianificazione_" + std::to_string(Nr) + ".txt");   // una riga per mappa: m, t FMM [ms], t Block FIM [ms]
        printf("\nRipianificazione (N = %d): tempo per ricalcolare il FM2 (passata 1 + passata 2) dopo ogni cambio di mappa\n", Nr);
        printf("  mappa                       u(A) FMM   u(A) BFIM   max|FMM-BFIM|   t FMM [ms]   t BFIM [ms]   speedup\n");
        const char* nomi[2] = { "0 pannello a sinistra", "1 pannello a destra" };
        for (int m = 0; m < 2; ++m) {
            imposta_mappa(m);
            const std::vector<unsigned char> mask = costruisci_maschera(Nr);
            const int iA = nodo(Nr, A.x, A.y, A.z);
            std::vector<float> u_fmm, u_bfim, F_fmm, F_bfim;
            double t_fmm = 1e30, t_bfim = 1e30;
            for (int r = 0; r < ripetizioni; ++r) {
                double t;
                if (r < ripetizioni_fmm) { fm2(false, Nr, mask, eps, niter, u_fmm, F_fmm, t);   t_fmm = std::min(t_fmm, t); }
                fm2(true,  Nr, mask, eps, niter, u_bfim, F_bfim, t); t_bfim = std::min(t_bfim, t);
            }
            float diff = 0.0f;
            for (size_t q = 0; q < u_fmm.size(); ++q) diff = fmaxf(diff, fabsf(u_fmm[q] - u_bfim[q]));
            printf("  %-26s   %8.4f   %9.4f   %13.2e   %10.1f   %11.2f   %7.1f\n",
                   nomi[m], u_fmm[iA], u_bfim[iA], diff, t_fmm, t_bfim, t_fmm / t_bfim);
            tr << m << " " << t_fmm << " " << t_bfim << "\n";
            scrivi_mappa(Nr, m, u_bfim, mask, F_bfim);       // il drone del Block FIM e quello del FMM usano la stessa u
        }
        imposta_mappa(0);
    }
    return 0;
}


In [ ]:
!nvcc -O3 -std=c++17 -arch=sm_75 main.cu -o fim3d && ./fim3d

In [ ]:
# RIPIANIFICAZIONE CON UN OGGETTO MOBILE (usa i file scritti da main.cu): un muro centrale ha due aperture e un pannello
# ne chiude una alla volta, spostandosi sull'altra ogni PERIODO secondi. Due droni partono insieme da A: ogni volta che
# il pannello si sposta, ognuno ricalcola il FM2 (tempi veri misurati da main.cu) continuando a volare; se il pannello gli
# chiude la strada frena e aspetta. Sinistra: Block FIM (GPU); destra: FMM (CPU). La scia e' colorata con la velocita'.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.interpolate import RegularGridInterpolator

# 0) PARAMETRI
N = 385                                                             # griglia dei file scritti da main.cu (Nr)
METRI_PER_UNITA = 10.0                                              # il cubo [-1, 1]^3 e' una stanza di 20 m di lato
V_MAX = 2.0                                                         # velocita' massima del drone [m/s], dove F = 1
SEC_PER_U = METRI_PER_UNITA / V_MAX                                 # 1 unita' di u = 5 s di volo
T_PRIMO = 4.0                                                       # primo spostamento del pannello: a t = 4 s, mentre i droni si avvicinano al muro
PERIODO = 10.0                                                      # poi il pannello resta 10 s su un'apertura e passa all'altra (con 5 s il Block FIM
                                                                    # insegue un'apertura che si richiude prima che ci arrivi: provare per vedere)
DT_SIM = 0.05                                                       # passo della simulazione del volo [s]
T_MAX = 120.0                                                       # durata massima della simulazione [s]: chi non arriva entro T_MAX si ferma li'

V_PIENA = 0.95 * V_MAX                                              # "velocita' massima": almeno il 95% di V_MAX
DT_FRAME = 0.3                                                      # un fotogramma ogni 0.3 s di volo
MS_FRAME = 60                                                       # il Play mostra un fotogramma ogni 60 ms: animazione 0.3 / 0.06 = 5 volte piu' veloce del vero
BLU, ARANCIO, ROSSO = "#2a78d6", "#eb6834", "#c0392b"
h = 2.0 / (N - 1)
x = -1.0 + h * np.arange(N)

# 1) DATI: le due mappe (u da B, maschera, lentezza F, ostacoli) e i tempi di ricalcolo
def leggi_ambiente(nome):
    amb = {"SCATOLA": [], "SFERA": [], "FORO": []}
    for riga in open(nome):
        k, *v = riga.split(); v = [float(t) for t in v]
        if k in ("A", "B"): amb[k] = np.array(v)
        elif k == "MARGINE": amb[k] = v[0]
        else: amb[k].append(tuple(v))
    return amb
mappe = []
for m in range(2):
    t = f"{N}_mappa{m}"
    u = np.fromfile(f"fim_{t}.bin", dtype=np.float32).reshape(N, N, N).astype(np.float64)    # u[k, i, j] = u(z, y, x)
    libero = np.fromfile(f"mask_{t}.bin", dtype=np.uint8).reshape(N, N, N) == 1
    F = np.fromfile(f"F_{t}.bin", dtype=np.float32).reshape(N, N, N).astype(np.float64)
    mappe.append(dict(u=u, libero=libero, F=F, amb=leggi_ambiente(f"ambiente_mappa{m}.txt"),
                      F_i=RegularGridInterpolator((x, x, x), F), peso=RegularGridInterpolator((x, x, x), libero.astype(float))))
A, B = mappe[0]["amb"]["A"], mappe[0]["amb"]["B"]
tempi = {}                                                          # tempo di ricalcolo [s] del FM2 per mappa e metodo
for riga in open(f"tempi_ripianificazione_{N}.txt"):
    m, t_fmm, t_bfim = riga.split()
    tempi[int(m)] = {"FMM": float(t_fmm) / 1000.0, "Block FIM": float(t_bfim) / 1000.0}

# 2) PERCORSO da un punto qualsiasi su una mappa: discesa lungo u (26 vicini) + accorciamento
mosse = [(a, b, c) for a in (-1, 0, 1) for b in (-1, 0, 1) for c in (-1, 0, 1) if (a, b, c) != (0, 0, 0)]
nodo = lambda p: tuple(np.round((np.asarray(p)[::-1] + 1.0) / h).astype(int))
coord = lambda n: -1.0 + h * np.array(n[::-1], dtype=float)
def tempo(mp, p, q):                                                # tempo (unita' di u) lungo il segmento p-q: somma di F * ds
    t = np.linspace(0.0, 1.0, int(np.linalg.norm(q - p) / (h / 4)) + 2)[:, None]
    return np.sum(mp["F_i"]((p + (t[:-1] + t[1:]) / 2 * (q - p))[:, ::-1])) * np.linalg.norm(q - p) / (len(t) - 1)
def si_vede(mp, p, q):                                              # il segmento p-q non attraversa ostacoli
    t = np.linspace(0.0, 1.0, int(np.linalg.norm(q - p) / (h / 4)) + 2)[:, None]
    return np.all(mp["peso"]((p + t * (q - p))[:, ::-1]) > 0.999)
def percorso(mp, p0):
    u, libero, F = mp["u"], mp["libero"], mp["F"]
    n, nB = nodo(p0), nodo(B)
    if not libero[n]:                                               # il pannello si e' spostato sul drone: parte dal nodo libero piu' vicino
        lo = [max(c - 16, 0) for c in n]
        vicini = np.argwhere(libero[lo[0]:n[0] + 17, lo[1]:n[1] + 17, lo[2]:n[2] + 17]) + lo
        n = tuple(int(c) for c in vicini[np.argmin(np.sum((vicini - n) ** 2, axis=1))])
    if not libero[n] or u[n] >= 1e29:
        raise SystemExit(f"il drone in {np.round(p0, 2)} e' dentro un ostacolo o non raggiunge B: cambia la mappa")
    nodi = [n]
    while n != nB:
        vic = [(n[0] + a, n[1] + b, n[2] + c) for a, b, c in mosse]
        vic = [w for w in vic if all(0 <= t < N for t in w) and libero[w]]
        n = min(vic, key=lambda w: u[w] + 0.5 * (F[n] + F[w]) * h * np.linalg.norm(np.subtract(w, n)))
        nodi.append(n)
    P = [np.asarray(p0, float)] + [coord(w) for w in nodi[1:]]
    tn = np.concatenate([[0], np.cumsum([tempo(mp, P[i], P[i + 1]) for i in range(len(P) - 1)])])
    out, a = [P[0]], 0
    while a < len(P) - 1:
        b = len(P) - 1
        while b > a + 1 and not (si_vede(mp, P[a], P[b]) and tempo(mp, P[a], P[b]) <= tn[b] - tn[a]):
            b -= 1
        out.append(P[b]); a = b
    return np.array(out)

# 3) VOLO: la mappa vera cambia a T_PRIMO e poi ogni PERIODO secondi (il pannello passa all'altra apertura), uguale per i due droni.
# Ogni drone vola sul suo percorso; quando la mappa e' cambiata ricalcola il FM2 (tempo del suo metodo) e intanto continua
# a volare. Se il passo successivo entra nel pannello (o nel suo margine) frena e aspetta. Se la mappa cambia ancora mentre
# ricalcola, finisce il calcolo e poi ricomincia con la mappa di quel momento. (Se il pannello arriva mentre il drone e'
# gia' nell'apertura, il drone finisce il passaggio.)
mappa_vera = lambda t: 0 if t < T_PRIMO else (1 + int((t - T_PRIMO) // PERIODO)) % 2   # 0 = pannello a sinistra, 1 = a destra
def fitto(mp, p0):                                                  # percorso da p0 sulla mappa mp, a punti fitti, e ascisse s [unita']
    perc = percorso(mp, p0)
    s = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(perc, axis=0), axis=1))])
    sf = np.linspace(0, s[-1], max(int(s[-1] / (h / 2)), 2))
    return np.stack([np.interp(sf, s, perc[:, c]) for c in range(3)], axis=1), sf
def vola(metodo):
    t, pos, m_piano = 0.0, A.copy(), 0
    Pf, sf = fitto(mappe[0], A); s = 0.0                            # percorso iniziale, calcolato prima del decollo
    piani = [(0.0, Pf)]
    T, P, V = [0.0], [A.copy()], [V_MAX / mappe[0]["F_i"](A[::-1][None])[0]]
    K = [(0, 0)]                                                    # per ogni istante: (percorso seguito, punto raggiunto)
    ricalcolo, fermo, frenato, vecchi = [], [], False, 0
    occupato, t_pronto, m_calcolo, arrivato = False, None, None, False
    libero = lambda m, p: mappe[m]["peso"](p[::-1][None])[0] > 0.999
    while t < T_MAX:
        mv = mappa_vera(t)
        if occupato and t >= t_pronto:                              # il percorso nuovo e' pronto: riparte da dove si trova
            Pf, sf = fitto(mappe[m_calcolo], pos); s = 0.0; m_piano, occupato = m_calcolo, False
            piani.append((t, Pf))
            if mv != m_calcolo: vecchi += 1                          # nel frattempo il pannello si e' gia' spostato di nuovo
        if not occupato and mv != m_piano:                          # la mappa e' cambiata: si ricalcola sulla mappa di adesso
            occupato, m_calcolo, t_pronto = True, mv, t + tempi[mv][metodo]
            ricalcolo.append((t, t_pronto))
        v = V_MAX / mappe[m_piano]["F_i"](pos[::-1][None])[0]       # velocita' [m/s] sulla mappa del percorso: v = V_MAX / F
        s_nuovo = min(s + v / METRI_PER_UNITA * DT_SIM, sf[-1])     # avanza di v * dt lungo il percorso
        nuovo = np.array([np.interp(s_nuovo, sf, Pf[:, c]) for c in range(3)])
        if libero(mv, pos) and not libero(mv, nuovo):               # entrerebbe nel pannello: frenata di emergenza
            if not frenato: fermo.append([t, None]); frenato = True
            v = 0.0
        else:
            if frenato: fermo[-1][1] = t; frenato = False
            pos, s = nuovo, s_nuovo
        t += DT_SIM
        T.append(t); P.append(pos.copy()); V.append(v); K.append((len(piani) - 1, int(np.searchsorted(sf, s))))
        if s >= sf[-1] - 1e-12: arrivato = True; break              # arrivato in B
    if frenato: fermo[-1][1] = t
    ricalcolo = [(a, min(b, t)) for a, b in ricalcolo]
    return dict(T=np.array(T), P=np.array(P), V=np.array(V), K=K, piani=piani, fermo=[tuple(f) for f in fermo],
                ricalcolo=ricalcolo, vecchi=vecchi, arrivato=arrivato)
voli = {met: vola(met) for met in ("Block FIM", "FMM")}
colore = {"Block FIM": BLU, "FMM": ARANCIO}
chiaro = {"Block FIM": "#a9cbf0", "FMM": "#f7c3a8"}                 # sfumatura chiara dello stesso colore (lampeggio)
scena_di = {"Block FIM": "scene", "FMM": "scene2"}
print(f"il pannello si sposta a t = {T_PRIMO:.0f} s e poi ogni {PERIODO:.0f} s; ricalcolo del FM2: Block FIM {tempi[0]['Block FIM'] * 1000:.0f} ms, FMM {tempi[0]['FMM']:.1f} s")
arrivo = lambda v: f"arriva in B dopo {v['T'][-1]:.1f} s" if v["arrivato"] else f"non arriva in B entro {T_MAX:.0f} s"
for met, v in voli.items():
    print(f"{met:9s}: {arrivo(v)}; ricalcoli {len(v['ricalcolo'])} "
          f"(di cui {v['vecchi']} gia' vecchi quando pronti); fermo davanti al pannello {sum(b - a for a, b in v['fermo']):6.2f} s")
rit = voli["FMM"]["T"][-1] - voli["Block FIM"]["T"][-1]
if voli["FMM"]["arrivato"] and voli["Block FIM"]["arrivato"]: print(f"il drone del FMM arriva {rit:.2f} s dopo quello del Block FIM")

# 4) GEOMETRIA da disegnare: ogni ostacolo come parallelepipedi (il muro senza porta e finestra)
def pieni(b, fori):                                                 # la scatola b meno i fori: lista di parallelepipedi
    tagli = [sorted({b[2 * a], b[2 * a + 1]} | {min(max(f[2 * a + s], b[2 * a]), b[2 * a + 1]) for f in fori for s in (0, 1)}) for a in range(3)]
    out = []
    for x0, x1 in zip(tagli[0][:-1], tagli[0][1:]):
        for y0, y1 in zip(tagli[1][:-1], tagli[1][1:]):
            for z0, z1 in zip(tagli[2][:-1], tagli[2][1:]):
                c = ((x0 + x1) / 2, (y0 + y1) / 2, (z0 + z1) / 2)
                if not any(f[0] <= c[0] <= f[1] and f[2] <= c[1] <= f[3] and f[4] <= c[2] <= f[5] for f in fori):
                    out.append((x0, x1, y0, y1, z0, z1))
    for asse in (2, 1, 0, 2, 1, 0):                                 # unisce i pezzi vicini: pochi parallelepipedi grandi
        out.sort(key=lambda p: tuple(p[2 * a] for a in range(3) if a != asse) + (p[2 * asse],))
        uniti = []
        for p in out:
            q = uniti[-1] if uniti else None
            if q and all(q[2 * a:2 * a + 2] == p[2 * a:2 * a + 2] for a in range(3) if a != asse) and abs(q[2 * asse + 1] - p[2 * asse]) < 1e-12:
                uniti[-1] = tuple(p[2 * a + s] if a == asse and s == 1 else q[2 * a + s] for a in range(3) for s in (0, 1))
            else:
                uniti.append(p)
        out = uniti
    return out
def mesh(scatole, nome, col, opacita):                              # un solo Mesh3d con tutti i parallelepipedi
    X, Y, Z, I, J, K = [], [], [], [], [], []
    for b in scatole:
        o = len(X)
        X += [b[0], b[1], b[1], b[0], b[0], b[1], b[1], b[0]]; Y += [b[2], b[2], b[3], b[3]] * 2; Z += [b[4]] * 4 + [b[5]] * 4
        for i, j, k in [(0, 1, 2), (0, 2, 3), (4, 5, 6), (4, 6, 7), (0, 1, 5), (0, 5, 4), (1, 2, 6), (1, 6, 5), (2, 3, 7), (2, 7, 6), (3, 0, 4), (3, 4, 7)]:
            I.append(o + i); J.append(o + j); K.append(o + k)
    return go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K, color=col, opacity=opacita, name=nome, flatshading=True, hoverinfo="name")
a0, a1 = (mappe[m]["amb"] for m in range(2))
aperture = sorted(set(a0["FORO"]) | set(a1["FORO"]))               # le due aperture del muro
pannello_su = {m: [f for f in aperture if f not in mappe[m]["amb"]["FORO"]][0] for m in range(2)}   # l'apertura chiusa dal pannello
interni = [b for b in a0["SCATOLA"] if b[0] > -0.99 and b[1] < 0.99]   # i muri interni (le pareti della stanza no: c'e' il contorno)
fori_di = lambda b: [f for f in aperture if b[0] - 1e-9 <= f[0] and f[1] <= b[1] + 1e-9]   # le aperture di quel muro
def superficie(b, fori, nome, col, opacita):                       # solo le facce esterne del muro b con i suoi fori (niente facce interne)
    tagli = [sorted({b[2 * a], b[2 * a + 1]} | {min(max(f[2 * a + s], b[2 * a]), b[2 * a + 1]) for f in fori for s in (0, 1)}) for a in range(3)]
    n = [len(t) - 1 for t in tagli]
    pieno = np.zeros(n, bool)                                       # cella (ix, iy, iz) del muro: piena se non sta in un foro
    for ix in range(n[0]):
        for iy in range(n[1]):
            for iz in range(n[2]):
                c = [(tagli[a][i] + tagli[a][i + 1]) / 2 for a, i in enumerate((ix, iy, iz))]
                pieno[ix, iy, iz] = not any(f[0] <= c[0] <= f[1] and f[2] <= c[1] <= f[3] and f[4] <= c[2] <= f[5] for f in fori)
    X, Y, Z, I, J, K = [], [], [], [], [], []
    for ix, iy, iz in zip(*np.nonzero(pieno)):
        for asse in range(3):
            for verso in (0, 1):                                    # faccia verso il basso (0) o verso l'alto (1) lungo l'asse
                vic = [ix, iy, iz]; vic[asse] += 1 if verso else -1
                if 0 <= vic[asse] < n[asse] and pieno[tuple(vic)]: continue   # confina con un'altra cella piena: faccia interna
                lo = [tagli[a][i] for a, i in enumerate((ix, iy, iz))]; hi = [tagli[a][i + 1] for a, i in enumerate((ix, iy, iz))]
                q = lo[asse] if verso == 0 else hi[asse]
                u_, w_ = [a for a in range(3) if a != asse]
                o = len(X)
                for du, dw in [(0, 0), (1, 0), (1, 1), (0, 1)]:
                    v = [0.0] * 3; v[asse] = q; v[u_] = (lo[u_], hi[u_])[du]; v[w_] = (lo[w_], hi[w_])[dw]
                    X.append(v[0]); Y.append(v[1]); Z.append(v[2])
                I += [o, o]; J += [o + 1, o + 2]; K += [o + 2, o + 3]
    return go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K, color=col, opacity=opacita, name=nome, flatshading=True, hoverinfo="name")
def muri_mesh():                                                    # tutti i muri interni in un solo Mesh3d
    parti = [superficie(b, fori_di(b), "muri", "#9a9a9a", 0.35) for b in interni]
    X, Y, Z, I, J, K = [], [], [], [], [], []
    for m_ in parti:
        o = len(X); X += list(m_.x); Y += list(m_.y); Z += list(m_.z)
        I += [o + i for i in m_.i]; J += [o + j for j in m_.j]; K += [o + k for k in m_.k]
    return go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K, color="#9a9a9a", opacity=0.35, name="muri", flatshading=True, hoverinfo="name")
spigoli = []                                                        # i 12 spigoli della stanza
for a in (-1, 1):
    for b in (-1, 1):
        spigoli += [[(-1, a, b), (1, a, b)], [(a, -1, b), (a, 1, b)], [(a, b, -1), (a, b, 1)]]
stanza = go.Scatter3d(x=sum([[p[0][0], p[1][0], None] for p in spigoli], []), y=sum([[p[0][1], p[1][1], None] for p in spigoli], []),
                      z=sum([[p[0][2], p[1][2], None] for p in spigoli], []), mode="lines", line=dict(color="gray", width=2),
                      name="stanza (20 m)", hoverinfo="skip")
ab = go.Scatter3d(x=[A[0], B[0]], y=[A[1], B[1]], z=[A[2], B[2]], mode="markers+text", text=["  A (partenza)", "  B (arrivo)"],
                  marker=dict(size=8, color=["green", "black"]), textfont=dict(size=14), name="A e B")
def centro(b):                                                      # dove scrivere il nome di un ostacolo: sopra il centro
    return [(b[0] + b[1]) / 2], [(b[2] + b[3]) / 2], [b[5] + 0.08]

# 5) ANIMAZIONE 3D: a sinistra il drone del Block FIM, a destra quello del FMM (stessi istanti)
def pannello(t, scena):                                             # il pannello mobile, sull'apertura chiusa all'istante t
    b = pannello_su[mappa_vera(t)]
    cx, cy, cz = centro(b)
    mm = mesh([b], "pannello", ROSSO, 0.85); mm.update(scene=scena, showlegend=False)
    return [mm, go.Scatter3d(x=cx, y=cy, z=cz, mode="text", text=["pannello"], textfont=dict(size=13, color=ROSSO), scene=scena, showlegend=False)]
def posizione(met, t):
    v = voli[met]; T, P, V = v["T"], v["P"], v["V"]
    if t >= T[-1]: return P[-1], 0.0
    k = np.searchsorted(T, t, side="right"); a = (t - T[k - 1]) / max(T[k] - T[k - 1], 1e-12)
    return P[k - 1] + a * (P[k] - P[k - 1]), V[k - 1] + a * (V[k] - V[k - 1])
def drone(met, t):                                                  # scia colorata con la velocita', percorso previsto e drone
    v = voli[met]; T, P, V = v["T"], v["P"], v["V"]
    pos, vel = posizione(met, t)
    fermo = any(a <= t < b for a, b in v["fermo"])
    ricalcola = any(a <= t < b for a, b in v["ricalcolo"])          # mentre ricalcola il pallino lampeggia
    ip, ik = v["K"][min(int(np.searchsorted(T, t, side="right")) - 1, len(T) - 1)]   # percorso seguito e punto raggiunto
    resto = np.vstack([pos[None], v["piani"][ip][1][ik:]]) if not fermo and t < T[-1] else np.empty((0, 3))
    scia, vs = np.vstack([P[T <= t][::3], pos[None]]), np.append(V[T <= t][::3], vel)
    sc, c = scena_di[met], colore[met]
    return [go.Scatter3d(x=scia[:, 0], y=scia[:, 1], z=scia[:, 2], mode="lines", scene=sc, showlegend=False, name=f"scia {met}",
                         line=dict(color=vs, colorscale="Turbo", cmin=0.0, cmax=V_MAX, width=12,
                                   colorbar=dict(title="velocita'<br>[m/s]", x=0.47, len=0.6) if met == "Block FIM" else None,
                                   showscale=(met == "Block FIM"))),
            go.Scatter3d(x=resto[:, 0], y=resto[:, 1], z=resto[:, 2], mode="lines", scene=sc, showlegend=False,
                         line=dict(color="black", width=2, dash="dash"), name=f"percorso previsto {met}"),
            go.Scatter3d(x=[pos[0]], y=[pos[1]], z=[pos[2]], mode="markers", scene=sc, showlegend=False, name=f"drone {met}",
                         marker=dict(size=7, color=chiaro[met] if ricalcola and round(t / DT_FRAME) % 2 else c,   # lampeggia: un fotogramma si' e uno no
                         line=dict(color=c, width=5)))]
def stato(met, t):
    v = voli[met]
    if t >= v["T"][-1]: return "arrivato" if v["arrivato"] else f"non arrivato entro {T_MAX:.0f} s"
    if any(a <= t < b for a, b in v["fermo"]): return "FERMO davanti al pannello, ricalcola"
    if any(a <= t < b for a, b in v["ricalcolo"]): return f"ricalcola, vola sul percorso vecchio a {posizione(met, t)[1]:.1f} m/s"
    return f"in volo a {posizione(met, t)[1]:.1f} m/s"
ASSI = dict(range=[-1.0, 1.0], autorange=False, showticklabels=False, title="")
SCENA = dict(xaxis=ASSI, yaxis=ASSI, zaxis=ASSI, aspectmode="cube", camera=dict(eye=dict(x=1.2, y=-1.7, z=1.3)))
def statiche(scena):
    out = []
    for tr in [stanza, muri_mesh(), ab]:
        tr = go.Figure(tr).data[0]; tr.update(scene=scena, showlegend=False); out.append(tr)
    return out
def dati(t):
    out = []
    for met in ("Block FIM", "FMM"):
        sc = scena_di[met]
        out += pannello(t, sc) + drone(met, t)
    return out
def titolo(t):
    return (f"t = {t:4.1f} s   |   <span style='color:{BLU}'>Block FIM: {stato('Block FIM', t)}</span>   |   "
            f"<span style='color:{ARANCIO}'>FMM: {stato('FMM', t)}</span>")
t_fine = max(v["T"][-1] for v in voli.values())
istanti = np.append(np.arange(0.0, t_fine, DT_FRAME), t_fine)
fisse = statiche("scene") + statiche("scene2")
mobili = list(range(len(fisse), len(fisse) + len(dati(0.0))))
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]], horizontal_spacing=0.02,
                    subplot_titles=(f"Block FIM (GPU): {arrivo(voli['Block FIM'])}", f"FMM (CPU): {arrivo(voli['FMM'])}"))
for tr in fisse + dati(0.0): fig.add_trace(tr)
fig.frames = [go.Frame(data=dati(t), traces=mobili, name=f"{t:.2f}", layout=dict(title=dict(text=titolo(t)))) for t in istanti]
fig.update_layout(title=dict(text=titolo(0.0), x=0.5), scene=SCENA, scene2=SCENA, height=920, margin=dict(l=0, r=0, t=80, b=290),
                  updatemenus=[dict(type="buttons", direction="left", x=0.08, y=0.0, xanchor="right", yanchor="top", pad=dict(r=10, t=60), buttons=[
                      dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=MS_FRAME, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                      dict(label="Pausa", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
                  sliders=[dict(x=0.1, len=0.88, y=0.0, yanchor="top", currentvalue=dict(prefix="tempo di volo t = ", suffix=" s"), pad=dict(t=40),
                                steps=[dict(label=f"{t:.0f}" if abs(t - round(t)) < 1e-9 else "", method="animate",
                                            args=[[f"{t:.2f}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for t in istanti])])
# Riquadro sotto l'animazione: come si passa dalla u del FM2 a metri e secondi
uA = mappe[0]["u"][nodo(A)]                                         # u(A) con il pannello a sinistra
spessore = (interni[0][1] - interni[0][0]) * METRI_PER_UNITA        # spessore di un muro interno [m]
fig.add_annotation(xref="paper", yref="paper", x=0.5, y=-0.27, xanchor="center", yanchor="top", align="left", showarrow=False,
                   bgcolor="#f4f6fa", bordercolor="#9a9a9a", borderwidth=1, borderpad=8, font=dict(size=13),
                   text=(f"<b>Dalla u del FM2 a metri e secondi</b><br>"
                         f"Scala: 1 unita' = {METRI_PER_UNITA:.0f} m, quindi la stanza [-1, 1]^3 e' {2 * METRI_PER_UNITA:.0f} x {2 * METRI_PER_UNITA:.0f} x {2 * METRI_PER_UNITA:.0f} m "
                         f"(muri spessi {spessore:.0f} m, margine di sicurezza {mappe[0]['amb']['MARGINE'] * METRI_PER_UNITA * 100:.0f} cm)<br>"
                         f"u = somma di F x tratto percorso: una distanza pesata (F = 1 lontano dagli ostacoli, F = 10 rasente)<br>"
                         f"Velocita' massima V_MAX = {V_MAX:.0f} m/s dove F = 1: tempo = u x {METRI_PER_UNITA:.0f} m / {V_MAX:.0f} m/s = u x {SEC_PER_U:.0f} s; "
                         f"velocita' v = V_MAX / F (da {V_MAX / 10:.1f} a {V_MAX:.0f} m/s)<br>"
                         f"Il pannello si sposta a {T_PRIMO:.0f} s e poi ogni {PERIODO:.0f} s | Esempio: pannello a sinistra, u(A) = {uA:.2f}, quindi {uA:.2f} x {SEC_PER_U:.0f} s = {uA * SEC_PER_U:.1f} s di volo | "
                         f"Play: 1 fotogramma = {DT_FRAME} s di volo ogni {MS_FRAME} ms, cioe' {DT_FRAME * 1000 / MS_FRAME:.0f} volte piu' veloce del vero"))
fig.write_html("oggetto_mobile.html", include_plotlyjs="cdn")    # anche come file da scaricare: si apre nel browser
fig.show()

# 6) VELOCITA' NEL TEMPO e TEMPO A VELOCITA' MASSIMA dei due droni
g = make_subplots(rows=1, cols=2, column_widths=[0.68, 0.32], horizontal_spacing=0.1,
                  subplot_titles=("Velocita' nel tempo (fasce = ricalcola; linee = il pannello si sposta)", "Come passa il tempo di volo"))
quote = {}
for met, v in voli.items():
    T, V = v["T"], v["V"]
    g.add_trace(go.Scatter(x=T, y=V, mode="lines", line=dict(color=colore[met], width=2.5), name=f"{met}: {arrivo(v)}"), row=1, col=1)
    for a, b in v["ricalcolo"]:
        g.add_vrect(x0=a, x1=max(b, a + 0.05), fillcolor=colore[met], opacity=0.2, line_width=0, row=1, col=1)
    dt, vm = np.diff(T), (V[:-1] + V[1:]) / 2
    quote[met] = {"a velocita' massima": dt[vm >= V_PIENA].sum(), "piu' lento": dt[(vm > 0.01) & (vm < V_PIENA)].sum(),
                  "fermo davanti al pannello": dt[vm <= 0.01].sum()}
for k in range(int((t_fine - T_PRIMO) // PERIODO) + 1):           # a T_PRIMO e poi ogni PERIODO secondi il pannello cambia apertura
    g.add_vline(x=T_PRIMO + k * PERIODO, line=dict(color=ROSSO, dash="dot", width=1), row=1, col=1)
g.add_hline(y=V_MAX, line=dict(color="gray", dash="dash"), annotation_text="V_MAX", row=1, col=1)
for voce, col in [("a velocita' massima", "#2ca02c"), ("piu' lento", "#bcbd22"), ("fermo davanti al pannello", "#7f7f7f")]:
    g.add_trace(go.Bar(x=list(quote), y=[quote[m][voce] for m in quote], name=voce, marker_color=col,
                       text=[f"{quote[m][voce]:.1f} s" for m in quote], textposition="inside"), row=1, col=2)
g.update_layout(template="plotly_white", barmode="stack", height=430, legend=dict(orientation="h", y=-0.2),
                title=(f"Il drone del FMM arriva {rit:.1f} s dopo: " if voli["FMM"]["arrivato"] else f"Il drone del FMM non arriva entro {T_MAX:.0f} s: ") + f"{quote['FMM']['fermo davanti al pannello']:.1f} s fermo davanti al pannello, contro {quote['Block FIM']['fermo davanti al pannello']:.2f} s")
g.update_xaxes(title_text="tempo di volo [s]", row=1, col=1); g.update_yaxes(title_text="velocita' [m/s]", row=1, col=1)
g.update_yaxes(title_text="tempo [s]", row=1, col=2)
g.write_html("oggetto_mobile_velocita.html", include_plotlyjs="cdn")
g.show()
for met, q in quote.items():
    print(f"{met:9s}: " + " | ".join(f"{k} {v:5.1f} s" for k, v in q.items()))
